# Qwen3 LoRA và đầu hồi quy
Train và suy luận là hai thao tác riêng. Dữ liệu mẫu lấy từ train, không phải holdout mới.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd() if (Path.cwd()/'career_ai').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))

In [2]:
from pathlib import Path
import json
import pandas as pd
state=json.loads((ROOT/'outputs/auto_dev/state.json').read_text())
selected=ROOT/state['selected_run']
metadata=json.loads((selected/'llm.json').read_text())
display(metadata)
display(pd.DataFrame(json.loads((selected/'llm_history.json').read_text())))
RUN_TRAINING=False
if RUN_TRAINING:
    if state['status'] != 'completed':
        raise RuntimeError('Goal vẫn hoạt động; không mở thêm thí nghiệm ngoài quota.')
    from datetime import datetime, timezone
    from career_auto.coordinator import training_processes
    from career_auto.training import train
    if training_processes():
        raise RuntimeError('Đang có tiến trình train khác.')
    config=dict(rank=metadata['rank'],learning_rate=metadata.get('learning_rate',5e-5),epochs=1)
    run=ROOT/'outputs'/('manual_notebook_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
    train(config,run)


{'dataset_manifest_sha256': '0b8aa65eb03f0516f568ae2eead7fc7ee1158e71a1ac5f9f0c1bbc3be6e72ff5',
 'preset': 'hf://keras/qwen3_1.7b_en',
 'base_source': {'repo': 'keras/qwen3_1.7b_en',
  'revision': '6938c40886582c96128ccde561bd8d1d69239a15',
  'local_path': 'models/qwen3',
  'license': 'Apache-2.0 per model card',
  'download_complete': True,
  'bytes': 6883417048,
  'sha256': 'c600f82e6d3b5c2c08d666b8d7fb961745e6265b10cea5ae92cbbe55645dd23a'},
 'rank': 8,
 'context': 1024,
 'epochs': 2,
 'training_rows': 10417,
 'limited_trial': False,
 'language_finetuned': True,
 'language_steps': 648,
 'selected_epoch': 1,
 'selection': 'market_mean_log_mae + 0.1 * validation_answer_ce',
 'device': ['/device:CPU:0']}

,epoch,train_loss,validation_log_mae,validation_answer_ce,validation_language_examples,selection_score
0,1,0.150497,0.430596,0.141151,32,0.444711
1,2,0.091371,0.443336,0.120638,32,0.455400
